# Simple Logistic Regression – Breast Cancer Dataset

This notebook demonstrates **binary classification using Logistic Regression** with basic preprocessing and evaluation metrics.

**Target:** 0 = Benign, 1 = Malignant

In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

## 1. Load the Breast Cancer Dataset

In [ ]:
# Load the built-in Breast Cancer dataset
data = load_breast_cancer()

# X contains input features
X = data.data

# y contains the target variable
# Original sklearn labels: 0 = Malignant, 1 = Benign
# We reverse them: 1 = Malignant, 0 = Benign
y = 1 - data.target

print("Dataset shape:", X.shape)
print("Target names:", data.target_names)
print("Target meaning: 0 = Benign, 1 = Malignant")

## 2. Basic Data Checking

In [ ]:
# Convert features to a DataFrame for easy checking
df = pd.DataFrame(X, columns=data.feature_names)
df["target"] = y

# Display first five rows
display(df.head())
       
# Check dataset size
print("Shape:", df.shape)

# Check missing values
print("Total missing values:", df.isnull().sum().sum())

# Check class distribution
print("\nTarget distribution:")
print(df["target"].value_counts())

## 3. Separate Features (X) and Target (y)

In [ ]:
# Remove target column from the input features
X = df.drop("target", axis=1)

# Store target separately
y = df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 4. Train-Test Split

In [ ]:
# Split the data into 80% training and 20% testing
# random_state=42 makes the split reproducible
# stratify=y keeps the class proportions similar
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

## 5. Basic Preprocessing – Feature Scaling

StandardScaler converts the features to a similar scale. We fit the scaler only on training data and then apply it to both training and testing data.

In [ ]:
# Create the scaler
scaler = StandardScaler()

# Learn scaling parameters from training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same scaling to test data
X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")

## 6. Create and Train Logistic Regression Model

In [ ]:
# Create Logistic Regression model
model = LogisticRegression(max_iter=1000)

# Train the model using training data
model.fit(X_train_scaled, y_train)
       
print("Model training completed!")

## 7. Make Predictions

In [ ]:
# Predict class labels for test data
y_pred = model.predict(X_test_scaled)

# Get probability of class 1 (Malignant)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("First 10 predictions:", y_pred[:10])
print("First 10 probabilities:", y_prob[:10])

## 8. Calculate Evaluation Metrics

In [ ]:
# Calculate different evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

# Display the results
print("========== MODEL PERFORMANCE ==========")
print("Accuracy  :", round(accuracy, 4))
print("Precision :", round(precision, 4))
print("Recall    :", round(recall, 4))
print("F1 Score  :", round(f1, 4))
print("ROC-AUC   :", round(auc, 4))

### Meaning of the metrics
- **Accuracy:** Overall percentage of correct predictions.
- **Precision:** Out of predicted Malignant cases, how many were actually Malignant.
- **Recall:** Out of actual Malignant cases, how many were correctly identified.
- **F1-score:** Balance between precision and recall.
- **ROC-AUC:** Measures how well the model separates the two classes across thresholds.

## 9. Confusion Matrix

In [ ]:
# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)
       
print("Confusion Matrix:")
print(cm)

# Display it as a simple plot without requiring seaborn
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], data.target_names)
plt.yticks([0, 1], data.target_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")

# Write values inside cells
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")
       
plt.show()

## 10. Classification Report

In [ ]:
# Show precision, recall, F1-score and support for both classes
print(classification_report(
    y_test,
    y_pred,
    target_names=data.target_names
))

## 11. ROC Curve

In [ ]:
# Calculate False Positive Rate and True Positive Rate
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# Plot ROC curve
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.2f})")
       
# Random classifier reference line
plt.plot([0, 1], [0, 1], linestyle="--")
       
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
       
plt.show()

## 12. Final Summary

The complete workflow is:

**Load Dataset → Check Data → Train-Test Split → Feature Scaling → Train Logistic Regression → Predict → Evaluate using Accuracy, Precision, Recall, F1, Confusion Matrix and ROC-AUC.**